In [1]:
import argparse
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message
from std_msgs.msg import String
from geometry_msgs.msg import PoseStamped
import rosbag2_py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from control_msgs.msg import DynamicJointState
from builtin_interfaces.msg import Time
from std_msgs.msg import Header
from scipy.spatial.transform import Rotation as R

In [2]:
def get_metadata(input_bag: str):
    # Use to get topic names and counts.
    reader = rosbag2_py.SequentialReader()
    reader.open(
        rosbag2_py.StorageOptions(uri=input_bag, storage_id="mcap"),
        rosbag2_py.ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )
    return reader.get_metadata()

def read_messages(input_bag: str):
    # Use to iterate through data.
    reader = rosbag2_py.SequentialReader()
    reader.open(
        rosbag2_py.StorageOptions(uri=input_bag, storage_id="mcap"),
        rosbag2_py.ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )
#     print(reader.get_metadata())

    topic_types = reader.get_all_topics_and_types()

    def typename(topic_name):
        for topic_type in topic_types:
            if topic_type.name == topic_name:
                return topic_type.type
        raise ValueError(f"topic {topic_name} not in bag")

    while reader.has_next():
        topic, data, timestamp = reader.read_next()
        msg_type = get_message(typename(topic))
        msg = deserialize_message(data, msg_type)
        yield topic, msg, timestamp
    del reader

In [3]:
bag_path = "../../logging/2025_2_16_rosbags/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag_0.mcap"

metadata = get_metadata(bag_path)

# Use this to find out topics and their message counts. 
topics = {name.topic_metadata.name : name.message_count for name in metadata.topics_with_message_count}

for name, count in topics.items():
    print(name, ": ",  count)

/odom_in_vicon_frame :  0
/dynamic_joint_states :  1621
/rb_slip_controller/transition_event :  0
/rb_slip_controller/cmd_vel :  7091
/joint_state_broadcaster/transition_event :  0
/joint_states :  1622
/rb_slip_controller/odom :  1616
/tf :  2434
/robot_description :  1
/tf_static :  2
/diagnostics :  59
/joy :  880
/rosout :  1684
/right_motor_currents :  9366
/pico_imu :  7412
/parameter_events :  1
/left_motor_currents :  9694
/rb/cmd_vel :  0
/events/write_split :  0
/vrpn_mocap/wand_test/accel :  0
/vrpn_mocap/rb1/twist :  11912
/vrpn_mocap/rb1/accel :  11913
/vrpn_mocap/rb1/pose :  11912
/vrpn_mocap/wand_test/pose :  0
/vrpn_mocap/wand_test/twist :  0


In [6]:
def get_header_stamp(msg: Time):
    sec = msg.sec
    nanosec = msg.nanosec
    return (sec, nanosec)

In [8]:
bag_path = "../../logging/2025_2_16_rosbags/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag_0.mcap"
metadata = get_metadata(bag_path)
# Use this to find out topics and their message counts. 
topics = {name.topic_metadata.name : name.message_count for name in metadata.topics_with_message_count}

desired_topics = [
    '/dynamic_joint_states',
]

('/dynamic_joint_states', array([0., 0., 0., ..., 0., 0., 0.], shape=(1621,)))


In [9]:
# Use message header stamp not the rosbag stamp!
msgs = read_messages(bag_path)

first_message = None

# Look at the timestamps.
i = 0
for topic, msg, timestamp in msgs:
    if i >2:
        break
        
    # Set rosbag epoch time.
    if first_message == None and hasattr(msg, "header"): # Use for relative time calc.
        first_message = msg
        print("First timestamped message is on ", topic)
        bag_epoch_ns = first_message.header.stamp.sec * 1e9 + first_message.header.stamp.nanosec 
        print("Rosbag epoch time is: ", bag_epoch_ns,"nanosec, \t", bag_epoch_ns/1e9, "sec\n\n\n")
    
    # If this message is from a topic we are interested in.
    if topic in desired_topics:
        print(msg,'\n')
        
        # Bad? Losing precision in the floating point representation here.
        t = msg.header.stamp.sec + msg.header.stamp.nanosec * 1e9
        print(t, "Seconds since ~1970")
        
        if i >0:
            # calc float sec+nanosec relative to start. subtract offset as int data type to avoid error.
            t_relative_ns = (msg.header.stamp.sec - first_message.header.stamp.sec) * int(1e9) \
                + (msg.header.stamp.nanosec - first_message.header.stamp.nanosec)
            t_relative = t_relative_ns * 1e-9  # Convert to float seconds only after subtraction
            print("Relative time is: ", t_relative, "sec\t", t_relative_ns, "nanosec\n\n\n")

        i += 1
#         data[topic]

First timestamped message is on  /right_motor_currents
Rosbag epoch time is:  1.7397453417104315e+18 nanosec, 	 1739745341.7104316 sec



control_msgs.msg.DynamicJointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1739745342, nanosec=947747393), frame_id=''), joint_names=['fr_wheel_joint', 'br_wheel_joint', 'fl_wheel_joint', 'bl_wheel_joint'], interface_values=[control_msgs.msg.InterfaceValue(interface_names=['position', 'velocity', 'encoder_count', 'current'], values=[52.320078645061734, 0.0, 53959.0, 0.0]), control_msgs.msg.InterfaceValue(interface_names=['position', 'velocity', 'encoder_count', 'current'], values=[-51.70242653703704, 0.0, -53322.0, 0.0]), control_msgs.msg.InterfaceValue(interface_names=['position', 'velocity', 'encoder_count', 'current'], values=[40.37040112654321, 0.0, 41635.0, 0.0]), control_msgs.msg.InterfaceValue(interface_names=['position', 'velocity', 'encoder_count', 'current'], values=[-35.649289487654315, 0.0, -36766.0, 0.0])]) 

9.4

In [10]:

bag_path = "../../logging/2025_2_16_rosbags/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag/2025_2_16__17_35_4130RPMback_60RPMfront_v2.bag_0.mcap"
metadata = get_metadata(bag_path)
# Use this to find out topics and their message counts. 
topics = {name.topic_metadata.name : name.message_count for name in metadata.topics_with_message_count}

print(topics)

desired_topics = [
    '/dynamic_joint_states',
    '/vrpn_mocap/rb1/pose'
]

msgs = read_messages(bag_path)

metadata = get_metadata(bag_path)
# Use this to find out topics and their message counts. 
topics = {name.topic_metadata.name : name.message_count for name in metadata.topics_with_message_count}

joint_names = ['fr_wheel_joint', 'br_wheel_joint', 'fl_wheel_joint', 'bl_wheel_joint']
encoder_data = {
    'names' : joint_names,
    'index' : 0,
    'stamp': np.zeros(1621, dtype=np.float64),
    'encoder_count' : np.zeros((len(joint_names),1621), dtype=np.int64)
}

vicon_pose_vars = ['x','y','z','quat_x','quat_y','quat_z','quat_w']
# vrpn_mocap_rb1_pose = {k: np.zeros(11912, dtype=np.float64) for k in vicon_pose_vars}
vrpn_mocap_rb1_data={
    'index' : 0,
    'stamp': np.zeros(11912, dtype=np.float64),
    'pose' : np.zeros((len(vicon_pose_vars),11912), dtype=np.float64)
}

left_current_vars = ['front','back']
left_current_data = {
    'index' : 0,
    'stamp': np.zeros(9694, dtype=np.float64),
    'currents' : np.zeros((len(vicon_pose_vars),11912), dtype=np.float64)
}

right_current_vars = ['front','back']
right_current_data = {
    'index' : 0,
    'stamp': np.zeros(9366, dtype=np.float64),
    'currents' : np.zeros((len(vicon_pose_vars),11912), dtype=np.float64)
}


{'/odom_in_vicon_frame': 0, '/dynamic_joint_states': 1621, '/rb_slip_controller/transition_event': 0, '/rb_slip_controller/cmd_vel': 7091, '/joint_state_broadcaster/transition_event': 0, '/joint_states': 1622, '/rb_slip_controller/odom': 1616, '/tf': 2434, '/robot_description': 1, '/tf_static': 2, '/diagnostics': 59, '/joy': 880, '/rosout': 1684, '/right_motor_currents': 9366, '/pico_imu': 7412, '/parameter_events': 1, '/left_motor_currents': 9694, '/rb/cmd_vel': 0, '/events/write_split': 0, '/vrpn_mocap/wand_test/accel': 0, '/vrpn_mocap/rb1/twist': 11912, '/vrpn_mocap/rb1/accel': 11913, '/vrpn_mocap/rb1/pose': 11912, '/vrpn_mocap/wand_test/pose': 0, '/vrpn_mocap/wand_test/twist': 0}


# Read Messages Using Filters
The API docs for rosbag2_py don't exist in any form. There are stubs but that's all. The C++ docs are also not extant.
There are helpful examples but they don't show anything but basic use. It's necessary to read through through the source. There's not even docstring.

https://github.com/ros2/rosbag2/blob/0105eec1d0bf650280b32076234fee73236bca31/rosbag2_py/rosbag2_py/_storage.pyi#L89

In [ ]:
import rosbag2_py
# Create a filter object
filt = rosbag2_py.StorageFilter(topics=['/tf'])

# Create a sequential reader object and assign the filter to it.
reader = rosbag2_py.SequentialReader()
reader.set_filter(filt)

reader.


TypeError: open(): incompatible function arguments. The following argument types are supported:
    1. (self: rosbag2_py._reader.SequentialReader, arg0: rosbag2_storage::StorageOptions, arg1: rosbag2_cpp::ConverterOptions) -> None

Invoked with: <rosbag2_py._reader.SequentialReader object at 0x7bb41a34a0b0>